# A large scale analysis of conversation data across myriad spoken corpora

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
bse = .002257

In [3]:
DATA_LOC = 'data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(VIS_PATH, 'final-document.tsv')

In [4]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

In [5]:
# to rename the corpus correctly . . . 
def rename(x):
    result = x
    if x.startswith('se'):
        result = 'CORAAL'
        
    if x.startswith('call'):
        result = x.split('-')[0]
    
    if x.startswith('MICASE'):
        result = 'MICASE'
    
    if x.startswith('instruction'):
        result = x.split('-')[-1]
    
    if x.startswith('CABNC'):
        result = 'CABNC'
    
    return result

In [6]:
df_ = pd.read_table(FINAL_DOC, sep='\t')

In [7]:
df_['corpus'] = [rename(co) for co in tqdm(df_['corpus'])]

  0%|          | 0/22954906 [00:00<?, ?it/s]

In [8]:
df_.head()

,Hxy,nx,ny,turn_delta,self,dyad,corpus,conversation_id,resid
0,6.344819,20.0,6.0,1,False,10851,GCSAusE,GCSAusE-GCSAusE-01,0.322016
1,6.000488,20.0,8.0,2,False,10851,GCSAusE,GCSAusE-GCSAusE-01,0.001965
2,5.521488,20.0,13.0,4,False,10851,GCSAusE,GCSAusE-GCSAusE-01,-0.416337
3,6.057978,20.0,6.0,6,False,10851,GCSAusE,GCSAusE-GCSAusE-01,0.035175
4,6.198739,20.0,6.0,7,False,10851,GCSAusE,GCSAusE-GCSAusE-01,0.175936


## Vis & Other Tests

In [9]:
import seaborn as sns
import plotly.tools as tls
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [10]:
tracking_doc = os.path.join(VIS_PATH,'tracking.txt')

In [11]:
sub_data_df = []

f = open(tracking_doc, 'r')
examples = f.read().split('\n\n')
f.close()

for l in examples:
    if l:
        vals = [v.strip() for v in l.split('|')[1:]]
        vals += [vals[-1].replace('[', '').replace(']', '').replace("'", '')]
        sub_data_df += [{'corpus': vals[0], 'examples': vals[-1]}]

sub_data_df = pd.DataFrame(sub_data_df)
sub_data_df.head()

,corpus,examples
0,callhome,"callhome-4807, callhome-4705, callhome-6217, c..."
1,CORAAL,"se3-ag1-se3-ag1-se3_ag1_m_02a, se3-ag4-se3-ag4..."
2,MICASE,"callfriend-eng_n-4874, callfriend-eng_n-4504, ..."
3,callfriend,"callfriend-eng_n-6869, callfriend-eng_n-6015, ..."
4,CABNC,"CABNC-KBV-CABNC-KBV-KBVRE00G, CABNC-KE2-CABNC-..."


In [12]:
sub_data_df = sub_data_df.drop_duplicates(keep='last')
sub_data_df.head()

,corpus,examples
0,callhome,"callhome-4807, callhome-4705, callhome-6217, c..."
1,CORAAL,"se3-ag1-se3-ag1-se3_ag1_m_02a, se3-ag4-se3-ag4..."
2,MICASE,"callfriend-eng_n-4874, callfriend-eng_n-4504, ..."
3,callfriend,"callfriend-eng_n-6869, callfriend-eng_n-6015, ..."
4,CABNC,"CABNC-KBV-CABNC-KBV-KBVRE00G, CABNC-KE2-CABNC-..."


In [13]:
names = ['other', 'self']
main_line_colors = ['#1f77b4','#ff7f0e']
ribbon_colors = ['rgba(31, 119, 180, 0.2)', 'rgba(255, 127, 14, 0.2)']

In [14]:
def create_subplot(
        dataframe, 
        new_fig, 
        row, 
        col, 
        names=names, 
        main_line_colors=main_line_colors, 
        ribbon_colors=ribbon_colors
):
    
    ax = sns.lineplot(
        data=dataframe, 
        y='resid', 
        x='turn_delta', 
        hue='self',
        legend=False
    )
    
    fig = tls.mpl_to_plotly(ax.figure)
    
    for i,trace in enumerate(fig.data):
        trace['name'] = names[i]
        # print(trace)
        trace['showlegend'] = False
        if (row==1) and (col==1):
            trace['showlegend'] = True
        trace['line']['color'] = main_line_colors[i]
        trace['legendgroup'] = names[i]
        # trace['legendgrouptitle'] = names[i]
        new_fig.add_trace(trace, row=row, col=col)
        
        # upper bounds for ribbon
        new_fig.add_trace(
            go.Scatter(
                x=trace['x'],
                y=(np.array(trace['y']) + bse*2).tolist(),
                mode='lines',
                line=dict(color=main_line_colors[i],width=0),
                name=names[i]+'_upper_bound',
                showlegend=False,
                legendgroup=names[i]
            ), row=row, col=col
        )
        
        # lower bounds for ribbon
        new_fig.add_trace(
            go.Scatter(
                x=trace['x'],
                y=(np.array(trace['y']) - bse*2).tolist(),
                mode='lines',
                line=dict(color=main_line_colors[i],width=0),
                name=names[i]+'_lower_bound',
                fill='tonexty',
                fillcolor=ribbon_colors[i],
                showlegend=False,
                legendgroup=names[i]
            ), row=row, col=col
        )

#### Actually creating the figure

In [15]:
conversation_id_col = 'conversation_id'

In [16]:
k_conversations = 12
k_columns = 3
k_rows = int(np.ceil(k_conversations/k_columns))

In [17]:
assignments = sum([[(row+1,col+1) for col in range(k_columns)] for row in range(k_rows)], [])

### Specific Corpus Visualization

In [19]:
for row in sub_data_df.index:
    print(SPECIFIC_CORPUS)
    SPECIFIC_CORPUS = sub_data_df['corpus'].loc[row]
    subdata = sub_data_df['examples'].loc[row].split(', ')

    subplot_fig = make_subplots(
        rows=k_rows, cols=k_columns, 
        #x_title=SPECIFIC_CORPUS,
        # subplot_titles=subdata
        # row_titles=['Zoom', 'Telephone', 'Instruction', 'BNC/Naturalistic'],
        # row_titles=['Instruction','Instruction', 'BNC/Naturalistic','BNC/Naturalistic'],
    )
    
    for (i, conversation) in tqdm(enumerate(subdata)):
        df__ = df_.loc[
            df_[conversation_id_col].isin([conversation])
            & (df_['turn_delta'] > 0)
            & (df_['turn_delta'] < 40)
        ]
        
        create_subplot(
            dataframe=df__,
            new_fig=subplot_fig,
            row=assignments[i][0],
            col=assignments[i][1]
        )   
        
        subplot_fig.update_layout(template='plotly')
        subplot_fig.write_html(os.path.join(VIS_PATH, 'ind_corpora', 'entropy-delta-individual-conversations-{}.html'.format(SPECIFIC_CORPUS)))

callfriend


0it [00:00, ?it/s]

callhome


0it [00:00, ?it/s]

CORAAL


...

0it [00:00, ?it/s]

CANDOR


0it [00:00, ?it/s]

GCSAusE


0it [00:00, ?it/s]